# 04 — Build gold

The research uses three complete hourly SMARD CSV exports for local calendar year 2025. Computation takes place in Microsoft Fabric against `lh_power_renewables`. Notebook boundaries are persisted data contracts rather than shared Python session variables.

Original Fabric execution results are preserved separately in `evidence`. The current execution status of each refactored notebook is recorded in `notebook_status.json`.

Current implemented scope: publish the validated calendar from `silver.dim_time_hour` to `gold.dim_time_hour`. No analytical fact tables or monthly aggregates are finalized yet. Their design follows the exploratory analysis and metric agreement; this notebook does not pretend those models already exist.

The input is `silver.dim_time_hour`. Publication has no dependency on exploratory session variables or charts.

In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
calendar = spark.table("silver.dim_time_hour")

calendar_checks = calendar.agg(
    F.count("*").alias("rows"),
    F.countDistinct("time_key").alias("unique_keys"),
    F.countDistinct("interval_start_utc").alias("unique_utc_starts"),
).first().asDict()

assert calendar_checks["rows"] == 8760, "Unexpected calendar coverage"
assert calendar_checks["unique_keys"] == 8760, "Duplicate calendar keys"
assert calendar_checks["unique_utc_starts"] == 8760, "Duplicate UTC starts"

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
calendar.write.format("delta").mode("overwrite").saveAsTable("gold.dim_time_hour")

saved_calendar = spark.table("gold.dim_time_hour")
assert saved_calendar.count() == 8760, "Unexpected saved calendar coverage"
print("Saved gold.dim_time_hour: 8760 rows")